In [1]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"


In [ ]:
import streamlit as st
from PyPDF2 import PdfReader 
pdf_folder="C:/Users/bhara/Desktop/MCQ-Generator/pdfs/MOOCS_DPM_SPM_Lect1.pdf"

all_text=""                                  
reader=PdfReader(pdf_folder)
for page in reader.pages:
    text=page.extract_text()
    all_text =all_text+text


In [ ]:
pages=[]   
reader=PdfReader(pdf_folder)
for page in reader.pages:
    text=page.extract_text()
    pages.append(text)

In [4]:
import re
print(len(all_text))
filtered_text=re.sub(r"NPTEL","",all_text)
filtered_text=re.sub(r"\d{1,2}+","",filtered_text)
print(len(filtered_text))

5242
5213


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=0)
chunks=splitter.create_documents([filtered_text])   # documents are more valid for retrieval process.


In [10]:
context=""
for i in chunks:
    context+=i.page_content
print(context)


Course Name: Software Project Management 
Faculty Name: Dr. DurgaPrasad MohapatraDepartment  : Computer Sc. & Engg. NIT Rourkela
TOPIC: Project Evaluation and 
ProgrammeManagement Business case for a project
Project portfolio management
Project evaluationThebusinesscase•Feasibilitystudyisalsocalledas“businesscase”
•Main focus is to determine whether it would be
financially&technicallyfeasibletodeveloptheS/W.
•Providesajustificationforstartingtheproject
•Should show that the benefits of the project willexceed the costs of development, implementationandoperation(orproduction)
•NeedstotakeaccountofbusinessrisksTypesoffeasibilitystudy
•TechnicalFeasibility
•FinancialFeasibility
•OperationalFeasibilityContents of a business case / feasibility study
. Introduction/ 
background
. The proposed 
project. The market. Organizational and 
operational infrastructure. The benefits
. Outline implementation 
plan
. Costs. The financial case. Risks. Management planContents of a business case     con

In [11]:
from pydantic import BaseModel, Field
from typing import List, Literal


class MCQ(BaseModel):
    question: str = Field(description="Multiple-choice question based on the PDF")
    options: List[str] = Field(
        description="four or More options labeled A, B, C, D, E, F",
        min_length=4,
        max_length=6
    )
    correct_answer: Literal["A", "B", "C", "D","E","F"] = Field(
        description="Label of the correct options"
    )
    explanation: str = Field(
        description="Brief explanation based only on the PDF"
    )
    topic: str = Field(description="Topic or section of the PDF")
    page_number: int | None = Field(
        default=None,
        description="PDF page number, if identifiable"
    )


class MCQSet(BaseModel):
    questions: List[MCQ] = Field(
        description="Topic-wise MCQs covering the important concepts in the PDF"
    )

In [12]:
mcq_prompt='''Act as an expert Software Project Management (SPM) examiner and learning assistant. Read the entire uploaded PDF and generate MCQs that help me understand, revise, and memorize its theory efficiently.

    Rules:
    1.  Cover all important concepts, definitions, principles, processes, advantages, disadvantages, characteristics, classifications, steps, comparisons, and examples mentioned in the PDF.
    2.  Generate as many high-quality MCQs as needed for comprehensive coverage without unnecessary repetition.
    3.  Give 4 or 5 options (A-D or E) per question, with exactly ONE or More correct answer.
    4.  Include a mix of definition-based, conceptual, application-based, comparison, and tricky exam-style questions.
    5.  Test important facts, differences between similar concepts, keywords, and easily forgotten details.
    6.  Use only information from the PDF strictly. Do not invent facts. If information is unclear, skip it.
    7.  Distribute correct answers randomly across A, B, C, and D or E. Avoid predictable patterns.
    8.  Use simple, clear language while preserving technical terminology from the PDF.
    9.  Divide questions into topic-wise sections, following the PDF's structure. Include page numbers when identifiable.
    10. Avoid duplicate questions and overly obvious distractors.
    11. Don't use your own knowledge to generate the questions and options.
    12. Generally generate the about 10 to 15 questions for each the provided context.
    
    Output format:
    - First, provide all questions with their four options.
    - Do NOT reveal answers immediately after questions.
    - At the end, provide an answer key with the correct option and a brief explanation based on the PDF.
    - Organize the output so I can attempt all questions before checking answers.
    context : {context}
    '''   

In [13]:
from langchain_core.prompts import PromptTemplate
prompt=PromptTemplate.from_template(mcq_prompt)

In [14]:
print(prompt)

input_variables=['context'] input_types={} partial_variables={} template="Act as an expert Software Project Management (SPM) examiner and learning assistant. Read the entire uploaded PDF and generate MCQs that help me understand, revise, and memorize its theory efficiently.\n\n    Rules:\n    1.  Cover all important concepts, definitions, principles, processes, advantages, disadvantages, characteristics, classifications, steps, comparisons, and examples mentioned in the PDF.\n    2.  Generate as many high-quality MCQs as needed for comprehensive coverage without unnecessary repetition.\n    3.  Give 4 or 5 options (A-D or E) per question, with exactly ONE or More correct answer.\n    4.  Include a mix of definition-based, conceptual, application-based, comparison, and tricky exam-style questions.\n    5.  Test important facts, differences between similar concepts, keywords, and easily forgotten details.\n    6.  Use only information from the PDF strictly. Do not invent facts. If inform

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,api_key="Your API key"
)

In [16]:
llm=llm.with_structured_output(MCQSet)
chain = prompt | llm
response = chain.invoke({"context":context})


In [22]:
print((response.questions[0]))
print((response.questions[0].correct_answer))
print(response.questions[0].explanation)

question='In the context of software project management, what is another term used for a "business case"?' options=['A. Project Charter', 'B. Feasibility Study', 'C. Requirements Specification', 'D. Design Document', 'E. Stakeholder Analysis'] correct_answer='B' explanation='The PDF states, "Feasibility study is also called as \'business case\'".' topic='Business Case' page_number=None
B
The PDF states, "Feasibility study is also called as 'business case'".


In [25]:
for i in range(len(response.questions)):
    print(response.questions[i].question)
    for j in response.questions[i].options:
        print(j)
    print("\n")

Which of the following is NOT identified as a symptom of the software crisis in the provided text?
A. Software often fails to meet user requirements.
B. Software is expensive.
C. Software is difficult to alter, debug, and enhance.
D. Software is often delivered late.
E. Software consumes no space, weight, or power.


According to the text, what is a key characteristic regarding software complexity and change?
A. The more changes made to a program, the less its complexity becomes.
B. The more complex software is, the easier it is to change.
C. Software complexity is unrelated to the ease of change.
D. The more changes made to a program, the greater its complexity becomes.


Which of the following is a key contributor to the software crisis, as stated in the document?
A. Decreasing skill shortage and manpower turnover.
B. High productivity improvements.
C. Poor project management.
D. Smaller problems in software development.


Why is Software Project Management considered hard, according

In [26]:
for j in range(len(response.questions)):
    with open("week-01-mcq_2026.txt","a") as f:
        f.write(f"Q{j+1}. ")
        f.write(response.questions[j].question)
        f.write("\n")
        for i in (response.questions[j].options):
            f.write(i)
            f.write("\n")
        f.write("\n\n")